In [ ]:
#| hide
%load_ext aimagics

# core

This page is rather short, as there are no public facing exports.

You can get help on the usage of `%ai` and `%%ai` with

In [ ]:
%ai?

Docstring:
Prompt an LLM as magic command.

- Can be used as 
  %ai prompt
- or as 
  %%ai prompt
  code 
  and other text in the cell
File:      ~/Documents/small-projects/aimagics-repo/aimagics/aimagics/core.py

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from IPython import get_ipython
from IPython.core.magic import magics_class, Magics, line_cell_magic
from IPython.display import clear_output, display, Markdown
import nbformat, ipynbname, os, warnings, requests, json
from traitlets import Unicode
from aimagics.systemprompt import sp
from aimagics.utils import split_into_cells, add_cells

In [ ]:
#| exporti
@magics_class
class AIMagics(Magics):
    """AIMagics class defining %ai line and %%ai cell magic. 
    
    Can be configured with
    %config
    %config AIMagis
    %config AIMagics.model = "google/gemini-3.8-flash"
    """

    # The config values
    model = Unicode(
        "~deepseek/deepseek-flash-latest",
        help="Model to be used."
    ).tag(config=True)

    system_prompt = Unicode(
        sp,
        help="The system prompt prepended to any prompt and context."
    ).tag(config=True)


    @line_cell_magic
    async def ai(self, line, cell=None):
        """Prompt an LLM as magic command.
        
        - Can be used as 
          %ai prompt
        - or as 
          %%ai prompt
          code 
          and any other text in the cell
        """
        line = line or ""
        prompt = line if cell is None else f"{line}\n{cell}".strip()
        try: agentsmd = open(os.path.join(os.path.dirname(ipynbname.path()), "AGENTS.md")).read()
        except: agentsmd = ""
        ctx = get_context() # TODO correct and better message history, TODO handle system prompt correctly, TODO tools
        final_prompt = "\n".join([self.system_prompt, agentsmd, ctx, prompt])

        h = display(Markdown('Processing request ...'), display_id=True) # Placeholder text
        accum = ""
        async for delta in stream_with_error_handling(final_prompt, self.model):
            accum += delta
            h.update(Markdown(accum))

        # post-processes md: strips code and adds as code cells at the end, with references from the md
        md, cs = split_into_cells(accum)
        clear_output(wait=True)
        display(Markdown("".join(md)))
        add_cells(cs)

In [ ]:
#| exporti
def line_magic_quotes(lines):
    """For line magic, add quotes if a ? is present. This prevents triggering ipython's help with ?"""
    magic_string = "%ai"
    if not lines or not lines[0].lstrip().split()[0] == magic_string: return lines
    arg = lines[0].lstrip()[len(magic_string):].strip() # the part after %ai stripped from whitespaces
    if "?" in arg:
        arg = arg.strip("'\"") # remove any " or ' at beginning or end
        line = f"{magic_string} '{arg}'\n" # build up new line magic with quoted input
        lines[0] = line 
        return lines
    else: 
        return lines

In [ ]:
#| exporti
def cell_magic_dummy_character(lines):
    """For cell magic (%%ai), don't need to add quotes, 
    but need to add a dummy character in case cell is empty"""
    magic_string = "%%ai"
    if not lines or not lines[0].lstrip().split()[0] == magic_string: return lines
    if len(lines) == 1:
        lines.append("\n")
    return lines

In [ ]:
#| exporti
def get_context():
    """Get notebook cells up to and including the calling cell.

    - Relies on state of notebook on disk. Recommended to turn autosave on.
    - If multiple cells have the same source, cells only up and including the first one are included.
    - Currently just uses the raw text content (json) of the current ipynb notebook."""
    ip = get_ipython()
    source_of_calling_cell = ip.kernel.get_parent()['content']['code']
    def normalize(s):
        return s.strip().replace("'", "").replace('"', "") # becomes fuzzy matching basically
    normalized_source = normalize(source_of_calling_cell)

    # There seems to be a vscode issue with autosaving that notebook cell id's are not assigned fast enough after modifying and executing a cell
    # https://github.com/jupyter/nbformat/blob/4346f97c9d435f41ddfe76d28758b23796fdcf5a/nbformat/validator.py#L346
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        nb = nbformat.read(ipynbname.path(), as_version=4)
        
    for i, cell in enumerate(nb['cells']):
        # there may be an error matching cells that have been modified by line_magic_quotes / cell_magic_dummy_character
        if cell['source'] == source_of_calling_cell or normalize(cell['source']) == normalized_source:
            break
    ctx = "\n".join([repr(cell) for cell in nb['cells'][:i+1]])
    return ctx

In [ ]:
#| exporti
async def stream_with_error_handling(prompt, model):
    response = requests.post(
        'https://openrouter.ai/api/v1/responses',
        headers={'Authorization': f'Bearer {os.environ['OPENROUTER_API_KEY']}'},
        json={'model': model, 'input': prompt, 'stream': True},
        stream=True
    )

    # Check initial HTTP status for pre-stream errors
    if response.status_code != 200:
        error_data = response.json()
        yield f"Error: {error_data['error']['message']}"
        
    # Process stream and handle mid-stream errors
    for line in response.iter_lines():
        if line:
            line_text = line.decode('utf-8')
            if line_text.startswith('data: '):
                data = line_text[6:]
                if data == '[DONE]': break
                try:
                    parsed = json.loads(data)
                    # Check for mid-stream errors
                    if parsed.get('type') == 'response.failed':
                        yield f"Stream error: {parsed['response']['error']['message']}\nStream terminated due to error\n"
                    if parsed.get('type') == 'response.incomplete':
                        details = parsed['response'].get('incomplete_details') or {}
                        reason = details.get('reason', 'unknown reason')
                        yield f"Stream incomplete: {reason}\n"
                    if parsed.get('type') == 'error':
                        yield f"Stream error: {parsed['message']}\n"
                    # Process normal content
                    if parsed.get('type') == 'response.output_text.delta':
                        yield parsed['delta']
                except json.JSONDecodeError: pass

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()